<a href="https://colab.research.google.com/github/crehimli5-jpg/Galen-AI/blob/main/GalenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install lightgbm shap optuna scikit-learn pandas numpy matplotlib seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 22.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
dosya_yolu = '/content/variant_summary.txt.gz'

df = pd.read_csv(
    dosya_yolu,
    sep='\t',
    compression='gzip',
    low_memory=False,
    nrows=300000
)

print(f'Satır sayısı: {len(df):,}')
print(f'Sütun sayısı: {len(df.columns)}')
print(df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: '/content/variant_summary.txt.gz'

In [ ]:
print(df['ClinicalSignificance'].value_counts().head(20))

In [ ]:

patojenik = ['Pathogenic', 'Likely pathogenic', 'Pathogenic/Likely pathogenic']
benign = ['Benign', 'Likely benign', 'Benign/Likely benign']


df_temiz = df[df['ClinicalSignificance'].isin(patojenik + benign)].copy()


df_temiz['etiket'] = df_temiz['ClinicalSignificance'].apply(
    lambda x: 1 if x in patojenik else 0
)

print(f'Temiz veri: {len(df_temiz):,} satır')
print(df_temiz['etiket'].value_counts())
print(f'\nPatojenik oranı: {df_temiz["etiket"].mean():.1%}')

In [ ]:
from sklearn.preprocessing import LabelEncoder


sayisal = ['Start', 'Stop', 'NumberSubmitters']


kategorik = ['Type', 'ReviewStatus', 'Chromosome', 'Assembly', 'GeneSymbol', 'Origin']


le = LabelEncoder()
for s in kategorik:
    df_temiz[s + '_enc'] = le.fit_transform(
        df_temiz[s].fillna('bilinmiyor').astype(str)
    )

enc = [s + '_enc' for s in kategorik]
ozellikler = sayisal + enc

X = df_temiz[ozellikler].fillna(-1)
y = df_temiz['etiket']

print(f'X boyutu: {X.shape}')
print(f'y dağılımı:\n{y.value_counts()}')

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    reg_lambda=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skorlar = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')

print(f'CV ROC-AUC skorları: {skorlar.round(4)}')
print(f'Ortalama: {skorlar.mean():.4f} ± {skorlar.std():.4f}')

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print(f'PR-AUC   : {average_precision_score(y_test, y_prob):.4f}')
print(f'F1 Skoru : {f1_score(y_test, y_pred):.4f}')
print(f'Duyarlılık: {recall_score(y_test, y_pred):.4f}')

In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=ax,
    name=f'Galen AI — LightGBM (AUC={0.9419:.3f})'
)
ax.plot([0,1],[0,1],'k--', label='Rastgele tahmin')
ax.set_title('ROC Eğrisi — Galen AI Modeli', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('roc_egrisi.png', dpi=150)
plt.show()
print('✓ roc_egrisi.png kaydedildi')

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[:500])


if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

plt.figure()
shap.summary_plot(
    sv,
    X_test.iloc[:500],
    feature_names=ozellikler,
    show=False
)
plt.title('SHAP — Özellik Önem Analizi')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ shap_summary.png kaydedildi')

In [ ]:

hatalar = X_test.copy()
hatalar['gercek'] = y_test.values
hatalar['tahmin'] = y_pred
hatalar['olasilik'] = y_prob
hatalar['yanlis'] = hatalar['gercek'] != hatalar['tahmin']

yanlis_df = hatalar[hatalar['yanlis'] == True]

print(f"Toplam hata: {len(yanlis_df):,}")
print(f"\nYanlış Negatif (Patojenik → Benign tahmin): {((hatalar['gercek']==1) & (hatalar['tahmin']==0)).sum():,}")
print(f"Yanlış Pozitif (Benign → Patojenik tahmin): {((hatalar['gercek']==0) & (hatalar['tahmin']==1)).sum():,}")


hatalar_tip = df_temiz.loc[yanlis_df.index, 'Type'].value_counts().head(10)
print(f"\nEn çok hata yapılan varyant tipleri:\n{hatalar_tip}")

In [ ]:
tekrar = df_temiz.duplicated().sum()
print(f"Tekrar eden kayıt sayısı: {tekrar}")

In [ ]:
from sklearn.metrics import f1_score, recall_score, confusion_matrix
import numpy as np

esikler = np.arange(0.3, 0.7, 0.05)
print("Eşik  | F1    | Duyarlılık | Özgüllük")
print("-" * 45)
for e in esikler:
    tahmin = (y_prob >= e).astype(int)
    f1 = f1_score(y_test, tahmin)
    recall = recall_score(y_test, tahmin)
    tn, fp, fn, tp = confusion_matrix(y_test, tahmin).ravel()
    ozgulluk = tn / (tn + fp)
    print(f"{e:.2f}  | {f1:.3f} | {recall:.3f}      | {ozgulluk:.3f}")

In [ ]:

print(df_temiz['Type'].value_counts())

In [ ]:

herediter_genler = ['BRCA1', 'BRCA2', 'TP53', 'PALB2', 'CHEK2', 'ATM']
df_herediter = df_temiz[df_temiz['GeneSymbol'].isin(herediter_genler)].copy()
print(f"Herediter Kanser: {len(df_herediter):,} varyant")
print(df_herediter['etiket'].value_counts())


df_pah = df_temiz[df_temiz['GeneSymbol'] == 'PAH'].copy()
print(f"\nPAH: {len(df_pah):,} varyant")
print(df_pah['etiket'].value_counts())


df_cftr = df_temiz[df_temiz['GeneSymbol'] == 'CFTR'].copy()
print(f"\nCFTR: {len(df_cftr):,} varyant")
print(df_cftr['etiket'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
import lightgbm as lgb

def panel_egit(df_panel, panel_adi, min_ornek=200):
    print(f"\n{'='*50}")
    print(f"PANEL: {panel_adi}")
    print(f"{'='*50}")

    X_p = df_panel[ozellikler].fillna(-1)
    y_p = df_panel['etiket']


    k = 3 if len(df_panel) < 2000 else 5


    sinif_agirlik = 'balanced' if y_p.value_counts().min() < 200 else None

    model_p = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=10,
        reg_lambda=0.3,
        class_weight=sinif_agirlik,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )


    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_p, y_p, test_size=0.2, random_state=42, stratify=y_p
    )


    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    cv_skorlar = cross_val_score(model_p, X_train_p, y_train_p,
                                  cv=cv, scoring='roc_auc')


    model_p.fit(X_train_p, y_train_p)
    y_prob_p = model_p.predict_proba(X_test_p)[:, 1]
    y_pred_p = (y_prob_p >= 0.5).astype(int)

    roc = roc_auc_score(y_test_p, y_prob_p)
    pr  = average_precision_score(y_test_p, y_prob_p)
    f1  = f1_score(y_test_p, y_pred_p)
    rec = recall_score(y_test_p, y_pred_p)

    print(f"CV ROC-AUC : {cv_skorlar.mean():.4f} ± {cv_skorlar.std():.4f}")
    print(f"ROC-AUC    : {roc:.4f}")
    print(f"PR-AUC     : {pr:.4f}")
    print(f"F1 Skoru   : {f1:.4f}")
    print(f"Duyarlılık : {rec:.4f}")

    return model_p, roc, pr, f1, rec


model_herediter, *s_herediter = panel_egit(df_herediter, "Herediter Kanser")
model_pah,      *s_pah       = panel_egit(df_pah,       "PAH")
model_cftr,     *s_cftr      = panel_egit(df_cftr,      "CFTR")

In [ ]:
import shap
import matplotlib.pyplot as plt

def panel_shap(model_p, df_panel, panel_adi):
    X_p = df_panel[ozellikler].fillna(-1)
    # En fazla 300 örnek al
    X_ornek = X_p.iloc[:300]

    explainer = shap.TreeExplainer(model_p)
    shap_values = explainer.shap_values(X_ornek)

    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values

    plt.figure()
    shap.summary_plot(sv, X_ornek, feature_names=ozellikler, show=False)
    plt.title(f'SHAP — {panel_adi} Paneli')
    plt.tight_layout()
    dosya = f'shap_{panel_adi.lower().replace(" ", "_")}.png'
    plt.savefig(dosya, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ {dosya} kaydedildi')

panel_shap(model_herediter, df_herediter, "Herediter Kanser")
panel_shap(model_pah,       df_pah,       "PAH")
panel_shap(model_cftr,      df_cftr,      "CFTR")

In [ ]:

ozellik_isimleri = ozellikler.copy()
print("Orijinal isimler:", ozellik_isimleri)


anonim_isimler = [f'f{i}' for i in range(len(ozellikler))]
print("Anonim isimler:", anonim_isimler)


X_anonim = X.copy()
X_anonim.columns = anonim_isimler

print("\nX_anonim ilk 3 satır:")
print(X_anonim.head(3))

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
import lightgbm as lgb


X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_anonim, y, test_size=0.2, random_state=42, stratify=y
)


model_anonim = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=50, reg_lambda=0.2, subsample=0.8,
    colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1
)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_skorlar = cross_val_score(model_anonim, X_train_a, y_train_a, cv=cv, scoring='roc_auc')
print(f'CV ROC-AUC: {cv_skorlar.mean():.4f} ± {cv_skorlar.std():.4f}')


model_anonim.fit(X_train_a, y_train_a)
y_prob_a = model_anonim.predict_proba(X_test_a)[:, 1]
y_pred_a = (y_prob_a >= 0.5).astype(int)

print(f'ROC-AUC  : {roc_auc_score(y_test_a, y_prob_a):.4f}')
print(f'PR-AUC   : {average_precision_score(y_test_a, y_prob_a):.4f}')
print(f'F1 Skoru : {f1_score(y_test_a, y_pred_a):.4f}')
print(f'Duyarlılık: {recall_score(y_test_a, y_pred_a):.4f}')

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer_a = shap.TreeExplainer(model_anonim)
shap_values_a = explainer_a.shap_values(X_test_a.iloc[:500])

if isinstance(shap_values_a, list):
    sv_a = shap_values_a[1]
else:
    sv_a = shap_values_a


grup_adlari = [
    'Genomik Pozisyon (Başlangıç)',
    'Genomik Pozisyon (Bitiş)',
    'Klinik Kanıt Birikimi',
    'Biyokimyasal Değişim Türü',
    'Klinik İnceleme Kalitesi',
    'Kromozom Grubu',
    'Referans Genomu',
    'Gen/Lokus Kimliği',
    'Varyant Kökeni',
]

plt.figure()
shap.summary_plot(sv_a, X_test_a.iloc[:500],
                  feature_names=grup_adlari, show=False)
plt.title('SHAP — Özellik Grup Analizi (Genel Model)')
plt.tight_layout()
plt.savefig('shap_genel_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ shap_genel_v2.png kaydedildi')

In [ ]:
def panel_shap_anonim(model_p, df_panel, panel_adi):
    X_p = df_panel[ozellikler].fillna(-1).copy()
    X_p.columns = anonim_isimler
    X_ornek = X_p.iloc[:300]

    explainer = shap.TreeExplainer(model_p)
    shap_values = explainer.shap_values(X_ornek)

    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values

    plt.figure()
    shap.summary_plot(sv, X_ornek, feature_names=grup_adlari, show=False)
    plt.title(f'SHAP — Özellik Grup Analizi ({panel_adi} Paneli)')
    plt.tight_layout()
    dosya = f'shap_{panel_adi.lower().replace(" ", "_")}_v2.png'
    plt.savefig(dosya, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ {dosya} kaydedildi')

panel_shap_anonim(model_herediter, df_herediter, "Herediter Kanser")
panel_shap_anonim(model_pah,       df_pah,       "PAH")
panel_shap_anonim(model_cftr,      df_cftr,      "CFTR")

In [ ]:
import lightgbm, sklearn, shap, pandas, numpy
print(f"lightgbm : {lightgbm.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"shap     : {shap.__version__}")
print(f"pandas   : {pandas.__version__}")
print(f"numpy    : {numpy.__version__}")

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
import lightgbm as lgb
import numpy as np


df_kucuk = df_temiz.sample(n=2800, random_state=42,
                            weights=df_temiz['etiket'].map({1:2000/98007, 0:800/72684}))

print("Simüle veri dağılımı:")
print(df_kucuk['etiket'].value_counts())

X_k = df_kucuk[ozellikler].fillna(-1).copy()
X_k.columns = anonim_isimler
y_k = df_kucuk['etiket']


configs = [
    {"isim": "Basit",   "num_leaves": 15, "n_estimators": 100, "min_child": 5},
    {"isim": "Orta",    "num_leaves": 31, "n_estimators": 200, "min_child": 10},
    {"isim": "Karmaşık","num_leaves": 63, "n_estimators": 300, "min_child": 20},
]

print(f"\n{'Model':<12} {'CV AUC':>10} {'CV F1':>10} {'CV MCC':>10}")
print("-" * 45)

for cfg in configs:
    model = lgb.LGBMClassifier(
        num_leaves=cfg["num_leaves"],
        n_estimators=cfg["n_estimators"],
        min_child_samples=cfg["min_child"],
        learning_rate=0.05,
        reg_lambda=0.3,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    auc_scores = cross_val_score(model, X_k, y_k, cv=cv, scoring='roc_auc')
    f1_scores  = cross_val_score(model, X_k, y_k, cv=cv, scoring='f1')
    mcc_scores = cross_val_score(model, X_k, y_k, cv=cv, scoring='matthews_corrcoef')

    print(f"{cfg['isim']:<12} {auc_scores.mean():.4f}±{auc_scores.std():.4f} "
          f"{f1_scores.mean():.4f}±{f1_scores.std():.4f} "
          f"{mcc_scores.mean():.4f}±{mcc_scores.std():.4f}")

In [ ]:

X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_k, y_k, test_size=0.2, random_state=42, stratify=y_k
)

model_orta = lgb.LGBMClassifier(
    num_leaves=31, n_estimators=200, min_child_samples=10,
    learning_rate=0.05, reg_lambda=0.3,
    class_weight='balanced', random_state=42, verbose=-1
)
model_orta.fit(X_train_k, y_train_k)
y_prob_k = model_orta.predict_proba(X_test_k)[:, 1]


from sklearn.metrics import f1_score, matthews_corrcoef

print("Eşik  | F1    | MCC   | Duyarlılık | Özgüllük")
print("-" * 55)
for esik in np.arange(0.20, 0.65, 0.05):
    tahmin = (y_prob_k >= esik).astype(int)
    f1  = f1_score(y_test_k, tahmin)
    mcc = matthews_corrcoef(y_test_k, tahmin)
    from sklearn.metrics import recall_score, confusion_matrix
    tn, fp, fn, tp = confusion_matrix(y_test_k, tahmin).ravel()
    duyarlilik = tp / (tp + fn)
    ozgulluk   = tn / (tn + fp)
    print(f"{esik:.2f}  | {f1:.3f} | {mcc:.3f} | {duyarlilik:.3f}      | {ozgulluk:.3f}")